# 🍑 Train ReDrafter for Qwen2.5-7B-Instruct

Trains a lightweight RNN draft head for speculative decoding on Apple Silicon.

**Requirements:** A100 GPU runtime (Runtime → Change runtime type → A100)

**Time:** ~2-3 hours on A100

**Output:** Drafter weights (~4.8GB) uploaded to HuggingFace Hub or downloaded

In [ ]:
# Step 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Step 2: Install dependencies
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
# Step 3: Clone momo-akira repo
import os
!git clone -b v2-token-level https://github.com/rdreilly58/momo-akira.git
os.chdir('/content/momo-akira')
print(f'Working directory: {os.getcwd()}')
!ls train_drafter.py

In [ ]:
# Step 4: Train the ReDrafter head
# This trains ONLY the tiny RNN drafter (~1.2B params).
# The 7B LLM is loaded frozen for generating hidden states.
#
# Expected: ~2-3 hours on A100, ~6-8 hours on T4

!cd /content/momo-akira && python train_drafter.py \
    --llm_name_or_path Qwen/Qwen2.5-7B-Instruct \
    --bf16 True \
    --output_dir ./drafter_output \
    --num_train_epochs 2 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 0.001 \
    --weight_decay 0.0 \
    --warmup_ratio 0.1 \
    --lr_scheduler_type cosine \
    --logging_steps 10 \
    --save_strategy steps \
    --save_steps 500 \
    --save_total_limit 2 \
    --evaluation_strategy no \
    --tf32 True \
    --model_max_length 2048 \
    --drafter_predict_n_tokens 5 \
    --drafter_num_layers 2 \
    --rnn True

In [ ]:
# Step 5: Check output
import os, glob

output_dirs = glob.glob('./drafter_output_redrafter_*')
if output_dirs:
    output_dir = output_dirs[0]
    print(f"Output directory: {output_dir}")
    for f in os.listdir(output_dir):
        size = os.path.getsize(os.path.join(output_dir, f))
        print(f"  {f}: {size/1e6:.1f} MB")
else:
    print("No output found! Check training logs above.")

In [ ]:
# Step 6: Quick validation — test the drafter produces reasonable logits
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json, glob

output_dir = glob.glob('./drafter_output_redrafter_*')[0]

# Load the config we saved
with open(f'{output_dir}/config.json') as f:
    drafter_cfg = json.load(f)
print(f"Drafter config: {json.dumps(drafter_cfg, indent=2)}")

# Quick forward pass test
print("\nLoading LLM for validation...")
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-7B-Instruct', trust_remote_code=True)
llm = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-7B-Instruct', torch_dtype=torch.bfloat16,
    device_map='auto', trust_remote_code=True
)

# Load drafter
import sys; sys.path.insert(0, '.')
from train_drafter import Drafter, DrafterConfig, ReDrafter
drafter = Drafter.from_pretrained(output_dir, torch_dtype=torch.bfloat16)
drafter = drafter.to(llm.device)

# Test
prompt = 'What is a hash table?'
inputs = tokenizer(prompt, return_tensors='pt').to(llm.device)
with torch.no_grad():
    outputs = llm.model(input_ids=inputs['input_ids'])
    h = outputs[0]  # hidden states
    # Get LLM's prediction
    llm_logits = llm.lm_head(h)
    llm_next = torch.argmax(llm_logits[0, -1]).item()
    print(f"LLM predicts: '{tokenizer.decode([llm_next])}'")

    # Get drafter's prediction (1 step)
    emb = llm.model.embed_tokens(inputs['input_ids'])
    last_emb = emb[:, -1:, :]
    state = torch.zeros_like(last_emb[:, 0, :])
    if drafter.config.rnn:
        state = torch.nn.functional.silu(
            drafter.rnn_w(last_emb[:, 0, :]) + drafter.rnn_u(state)
        )
    combined = torch.cat([h[:, -1:, :], state.unsqueeze(1)], dim=-1)
    if hasattr(drafter, 'input_proj'):
        combined = drafter.input_proj(combined)
    draft_logits = combined
    for layer in drafter.lm_head:
        draft_logits = layer(draft_logits)
    draft_next = torch.argmax(draft_logits[0, 0]).item()
    print(f"Drafter predicts: '{tokenizer.decode([draft_next])}'")
    print(f"Match: {'✅' if llm_next == draft_next else '❌ (expected — drafter approximates LLM)'}")

# Top-5 overlap
llm_top5 = set(torch.topk(llm_logits[0, -1], 5).indices.tolist())
draft_top5 = set(torch.topk(draft_logits[0, 0], 5).indices.tolist())
overlap = len(llm_top5 & draft_top5)
print(f"Top-5 overlap: {overlap}/5")

In [ ]:
# Step 7: Package and download
import shutil, glob

output_dir = glob.glob('./drafter_output_redrafter_*')[0]
archive_name = 'redrafter-qwen25-7b-drafter'

# Create clean package
os.makedirs(archive_name, exist_ok=True)
for f in ['config.json', 'model.safetensors']:
    src = os.path.join(output_dir, f)
    if os.path.exists(src):
        shutil.copy2(src, archive_name)
        print(f"Copied {f}")

# Also check for pytorch_model.bin
for f in glob.glob(os.path.join(output_dir, 'pytorch_model*')):
    shutil.copy2(f, archive_name)
    print(f"Copied {os.path.basename(f)}")

# Tar it up
shutil.make_archive(archive_name, 'gztar', '.', archive_name)
print(f"\n📦 Archive ready: {archive_name}.tar.gz")
print(f"Size: {os.path.getsize(f'{archive_name}.tar.gz')/1e6:.1f} MB")

In [ ]:
# Step 8: Download to local machine
from google.colab import files
files.download(f'{archive_name}.tar.gz')
print("\n✅ Download started!")
print("After download, transfer to your Mac:")
print(f"  tar xzf {archive_name}.tar.gz")
print(f"  mv {archive_name} ~/Projects/momo-akira/drafter_weights/")

## Alternative: Save to Google Drive
If download is slow, mount Drive and copy there instead.

In [ ]:
# Optional: Save to Google Drive instead of downloading
# from google.colab import drive
# drive.mount('/content/drive')
# !cp {archive_name}.tar.gz /content/drive/MyDrive/
# print("Saved to Google Drive!")